# 07 — Multi-Backbone Binary Classification

**Goal:** Train 4 additional transformer backbones on all 3 binary datasets.

**Backbones:** XLM-RoBERTa, mBERT, Sagorsarker BanglaBERT, MuRIL  
**Datasets:** Ben-Sarc, BanglaSarc, BanglaSarc3 (all binary)  
**Protocol:** epochs=3, lr=2e-5, batch=8, max_len=128  

**Disk‑safe:** 
- All caches and temporary files go to `/kaggle/tmp` (60 GB).
- Hugging Face cache is cleared after every run.
- Only prediction CSVs and metrics are kept; model weights are discarded.
- Never exceeds 19 GB of disk usage.

In [1]:
# Upgrade libraries to fix compatibility issues
!pip install --upgrade transformers huggingface_hub -q

In [2]:
import os, sys, json, time, warnings, gc, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

# ── Detect environment and set up paths ──
if os.path.exists('/kaggle/working'):
    REPO_ROOT = Path('/kaggle/working/Sarcasm_detection')
    BIG_TMP = Path('/kaggle/tmp')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '00_admin').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    BIG_TMP = REPO_ROOT / '_tmp'

PROJECT     = REPO_ROOT
SPLIT_DIR   = PROJECT / '01_data' / 'interim' / 'splits'
CKPT_DIR    = PROJECT / '03_models' / 'checkpoints'
TABLE_DIR   = PROJECT / '04_outputs' / 'tables'
PRED_DIR    = PROJECT / '04_outputs' / 'predictions'
LOG_DIR     = PROJECT / '04_outputs' / 'run_logs'
REPORT_DIR  = PROJECT / '04_outputs' / 'reports'

for d in [TABLE_DIR, PRED_DIR, LOG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR    = BIG_TMP / 'hf_cache'
TEMP_TRAIN_DIR  = BIG_TMP / 'train_cache'
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT}")
print(f"HuggingFace cache: {HF_CACHE_DIR}")
print(f"Temp training dir: {TEMP_TRAIN_DIR}")
print(f"Splits found: {len(list(SPLIT_DIR.glob('*.csv')))} files")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")

def disk_free_gb(path='/'):
    try:
        return shutil.disk_usage(path).free / 1e9
    except FileNotFoundError:
        # Fallback to current directory if path doesn't exist (e.g., /kaggle/working on local)
        return shutil.disk_usage('.').free / 1e9

def clear_hf_cache():
    if HF_CACHE_DIR.exists():
        shutil.rmtree(HF_CACHE_DIR, ignore_errors=True)
        HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"  Cleared HF cache (now {disk_free_gb(BIG_TMP):.1f} GB free on {BIG_TMP})")

if os.path.exists('/kaggle/working'):
    print(f"Kaggle working free: {disk_free_gb('/kaggle/working'):.1f} GB")
    print(f"Kaggle tmp free:     {disk_free_gb('/kaggle/tmp'):.1f} GB")
    clear_hf_cache()

Project root: /kaggle/working/Sarcasm_detection
HuggingFace cache: /kaggle/tmp/hf_cache
Temp training dir: /kaggle/tmp/train_cache
Splits found: 12 files
GPU available: True
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB
Kaggle working free: 20.9 GB
Kaggle tmp free:     1310.3 GB
  Cleared HF cache (now 1310.3 GB free on /kaggle/tmp)


In [3]:
BACKBONES = {
    'xlm-roberta':    'xlm-roberta-base',
    'mbert':          'bert-base-multilingual-cased',
    'sagorsarker-bb': 'sagorsarker/bangla-bert-base',
    'muril':          'google/muril-base-cased',
}

DATASETS = {
    'ben_sarc_binary': {
        'train': SPLIT_DIR / 'ben_sarc_binary_train.csv',
        'val':   SPLIT_DIR / 'ben_sarc_binary_val.csv',
        'test':  SPLIT_DIR / 'ben_sarc_binary_test.csv',
    },
    'banglasarc_binary': {
        'train': SPLIT_DIR / 'banglasarc_binary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc_binary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc_binary_test.csv',
    },
    'banglasarc3_binary': {
        'train': SPLIT_DIR / 'banglasarc3_binary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc3_binary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc3_binary_test.csv',
    },
}

MAX_LENGTH    = 128
EPOCHS        = 3
BATCH_SIZE    = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
SEED          = 42
METRIC_FOR_BEST = 'macro_f1'

TEXT_COL  = 'text'
LABEL_COL = 'label_binary'
SAVE_MODEL_WEIGHTS = False

print(f"Total runs: {len(BACKBONES) * len(DATASETS)}")
print(f"Save model weights: {SAVE_MODEL_WEIGHTS}")
print(f"All temporary data will be stored in: {BIG_TMP}")

Total runs: 12
Save model weights: False
All temporary data will be stored in: /kaggle/tmp


In [4]:
class SarcasmDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length',
                                   max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'binary_f1': f1_score(labels, preds, average='binary', pos_label=1),
        'precision': precision_score(labels, preds, average='binary', pos_label=1),
        'recall': recall_score(labels, preds, average='binary', pos_label=1),
    }

def load_split(csv_path):
    df = pd.read_csv(csv_path)
    return df[TEXT_COL].astype(str).tolist(), df[LABEL_COL].astype(int).tolist()

for ds_name, paths in DATASETS.items():
    tr, trl = load_split(paths['train'])
    te, tel = load_split(paths['test'])
    print(f"{ds_name}: train={len(tr)}, test={len(te)}, labels={sorted(set(trl))}")

ben_sarc_binary: train=20508, test=2564, labels=[0, 1]
banglasarc_binary: train=4089, test=512, labels=[0, 1]
banglasarc3_binary: train=6413, test=802, labels=[0, 1]


In [5]:
!pip install --upgrade transformers -q

In [6]:
def train_and_evaluate(model_tag, model_name, dataset_tag, dataset_paths):
    print(f"\n{'='*70}")
    print(f"  {model_tag} x {dataset_tag}  |  Model: {model_name}")
    if os.path.exists('/kaggle/working'):
        working_free = disk_free_gb('/kaggle/working')
    else:
        working_free = disk_free_gb('.')
    print(f"  Disk free (tmp): {disk_free_gb(BIG_TMP):.1f} GB | working: {working_free:.1f} GB")
    print(f"{'='*70}")
    t_start = time.time()

    train_texts, train_labels = load_split(dataset_paths['train'])
    val_texts, val_labels = load_split(dataset_paths['val'])
    test_texts, test_labels = load_split(dataset_paths['test'])
    print(f"  Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=str(HF_CACHE_DIR))
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, ignore_mismatched_sizes=True,
        cache_dir=str(HF_CACHE_DIR)
    )

    train_ds = SarcasmDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_ds = SarcasmDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
    test_ds = SarcasmDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

    run_tmp = TEMP_TRAIN_DIR / f"{model_tag}_{dataset_tag}"
    if run_tmp.exists():
        shutil.rmtree(run_tmp)
    run_tmp.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(run_tmp),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model=METRIC_FOR_BEST,
        greater_is_better=True,
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to='none',
        dataloader_num_workers=2,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # Use predict instead of evaluate to avoid callback issues
    val_output = trainer.predict(val_ds)
    val_metrics = val_output.metrics
    print(f"  Val — Acc: {val_metrics['test_accuracy']:.4f} | Macro-F1: {val_metrics['test_macro_f1']:.4f}")

    test_output = trainer.predict(test_ds)
    test_metrics = test_output.metrics
    test_preds = np.argmax(test_output.predictions, axis=-1)
    test_probs = torch.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()
    print(f"  Test — Acc: {test_metrics['test_accuracy']:.4f} | Macro-F1: {test_metrics['test_macro_f1']:.4f}")

    # Validation predictions (already from val_output)
    val_preds = np.argmax(val_output.predictions, axis=-1)
    val_probs = torch.softmax(torch.tensor(val_output.predictions), dim=-1).numpy()

    for split_name, texts, labels, preds, probs in [
        ('test', test_texts, test_labels, test_preds, test_probs),
        ('val',  val_texts,  val_labels,  val_preds,  val_probs),
    ]:
        pred_df = pd.DataFrame({
            'text': texts, 'true_label': labels, 'pred_label': preds,
            'prob_0': probs[:, 0], 'prob_1': probs[:, 1]
        })
        pred_df.to_csv(PRED_DIR / f"07_{model_tag}_{dataset_tag}_{split_name}_predictions.csv", index=False)

    if SAVE_MODEL_WEIGHTS:
        save_dir = CKPT_DIR / f"07_{model_tag}_{dataset_tag}" / 'best_model'
        save_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(save_dir))
        tokenizer.save_pretrained(str(save_dir))

    cm = confusion_matrix(test_labels, test_preds).tolist()
    t_elapsed = time.time() - t_start

    result = {
        'model_tag': model_tag, 'model_name': model_name, 'dataset': dataset_tag,
        'val_accuracy':    val_metrics['test_accuracy'],
        'val_macro_f1':    val_metrics['test_macro_f1'],
        'val_weighted_f1': val_metrics['test_weighted_f1'],
        'val_binary_f1':   val_metrics['test_binary_f1'],
        'test_accuracy':    test_metrics['test_accuracy'],
        'test_macro_f1':    test_metrics['test_macro_f1'],
        'test_weighted_f1': test_metrics['test_weighted_f1'],
        'test_binary_f1':   test_metrics['test_binary_f1'],
        'test_precision':   test_metrics['test_precision'],
        'test_recall':      test_metrics['test_recall'],
        'confusion_matrix': json.dumps(cm),
        'train_samples': len(train_texts),
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE,
        'max_length': MAX_LENGTH, 'time_seconds': round(t_elapsed, 1),
    }

    del model, trainer, train_ds, val_ds, test_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if run_tmp.exists():
        shutil.rmtree(run_tmp)
    clear_hf_cache()

    print(f"  Cleaned up. Disk free (tmp): {disk_free_gb(BIG_TMP):.1f} GB")
    return result

In [7]:
all_results = []
summary_path = TABLE_DIR / '07_multi_backbone_binary_summary.csv'

if summary_path.exists():
    prev_df = pd.read_csv(summary_path)
    all_results = prev_df.to_dict('records')
    done_keys = set(zip(prev_df['model_tag'], prev_df['dataset']))
    print(f"Resuming: {len(done_keys)} runs already completed")
else:
    done_keys = set()

if TEMP_TRAIN_DIR.exists():
    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
    TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
clear_hf_cache()

total_runs = len(BACKBONES) * len(DATASETS)
run_num = len(done_keys)

for model_tag, model_name in BACKBONES.items():
    for ds_tag, ds_paths in DATASETS.items():
        if (model_tag, ds_tag) in done_keys:
            print(f"Skipping {model_tag} x {ds_tag} (done)")
            continue
        run_num += 1
        print(f"\n>>> Run {run_num}/{total_runs}")
        
        # Only enforce disk space limits on Kaggle
        if os.path.exists('/kaggle/working'):
            free_working = disk_free_gb('/kaggle/working')
            if free_working < 3.0:
                print(f"WARNING: Only {free_working:.1f} GB free. Cleaning...")
                clear_hf_cache()
                if free_working < 2.0:
                    raise RuntimeError(f"Insufficient disk space ({free_working:.1f} GB). Aborting.")
        else:
            # Local run – just log available space
            free_local = disk_free_gb('.')
            print(f"  Local disk free: {free_local:.1f} GB")
        try:
            result = train_and_evaluate(model_tag, model_name, ds_tag, ds_paths)
            all_results.append(result)
            pd.DataFrame(all_results).to_csv(summary_path, index=False)
            print(f"  Saved. {total_runs - run_num} runs remaining.")
        except Exception as e:
            print(f"  FAILED: {e}")
            if all_results:
                pd.DataFrame(all_results).to_csv(summary_path, index=False)
            if TEMP_TRAIN_DIR.exists():
                shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
            clear_hf_cache()
            raise

if TEMP_TRAIN_DIR.exists():
    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
clear_hf_cache()

print(f"\n{'='*70}")
print(f"All {total_runs} runs complete!")
print(f"Disk free (working): {disk_free_gb('/kaggle/working'):.1f} GB")
print(f"Disk free (tmp):     {disk_free_gb(BIG_TMP):.1f} GB")

  Cleared HF cache (now 1310.3 GB free on /kaggle/tmp)

>>> Run 1/12

  xlm-roberta x ben_sarc_binary  |  Model: xlm-roberta-base
  Disk free (tmp): 1310.3 GB | working: 20.9 GB


  Train: 20508 | Val: 2564 | Test: 2564


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.encoder.layer.1.output.LayerNorm.beta', 'bert.encoder.layer.1.output.LayerNorm.gamma', 'bert.encoder.layer.2.attention.output.LayerNorm.beta', 'bert.encoder.layer.2.attention.output.LayerNorm.gamma', 'bert.encoder.layer.2.output.LayerNorm.beta', 'bert.encoder.layer.2.output.LayerNorm.gamma', 'bert.encoder.layer.3.attention.output.LayerNorm.beta', 'bert.encoder.layer.3.attention.output.LayerNorm.gamma', 'bert.encoder.layer.3.output.LayerNorm.beta', 'bert.encoder.layer.3.output.LayerNorm.gamma', 'bert.encoder.layer.4.attention.output.LayerNor

  Val — Acc: 0.7718 | Macro-F1: 0.7718


  Test — Acc: 0.7357 | Macro-F1: 0.7356


  Cleared HF cache (now 1310.3 GB free on /kaggle/tmp)
  Cleaned up. Disk free (tmp): 1310.3 GB
  Saved. 0 runs remaining.
  Cleared HF cache (now 1310.3 GB free on /kaggle/tmp)

All 12 runs complete!
Disk free (working): 20.9 GB
Disk free (tmp):     1310.3 GB


In [8]:
results_df = pd.read_csv(summary_path)
display_cols = ['model_tag', 'dataset', 'test_accuracy', 'test_macro_f1', 'test_binary_f1', 'test_precision', 'test_recall', 'time_seconds']
print("="*90)
print("  MULTI-BACKBONE BINARY RESULTS (Test Set)")
print("="*90)
print(results_df[display_cols].to_string(index=False, float_format='%.4f'))

  MULTI-BACKBONE BINARY RESULTS (Test Set)
     model_tag            dataset  test_accuracy  test_macro_f1  test_binary_f1  test_precision  test_recall  time_seconds
   xlm-roberta    ben_sarc_binary         0.7613         0.7613          0.7613          0.7613       0.7613     1880.8000
   xlm-roberta  banglasarc_binary         0.9473         0.9446          0.9323          0.9163       0.9490      399.3000
   xlm-roberta banglasarc3_binary         0.7307         0.7306          0.7353          0.7229       0.7481      608.7000
         mbert    ben_sarc_binary         0.7500         0.7496          0.7595          0.7317       0.7894     1430.6000
         mbert  banglasarc_binary         0.9473         0.9448          0.9330          0.9082       0.9592      301.8000
         mbert banglasarc3_binary         0.7282         0.7280          0.7212          0.7402       0.7032      461.3000
sagorsarker-bb    ben_sarc_binary         0.7516         0.7512          0.7601          0.7349 

In [9]:
pivot = results_df.pivot_table(index='model_tag', columns='dataset', values='test_macro_f1', aggfunc='first')

# Load previous baselines if available
bb_path = TABLE_DIR / '05_baseline_banglabert_binary_summary_all_datasets.csv'
if bb_path.exists():
    bb_df = pd.read_csv(bb_path)
    bb_row = {}
    for _, row in bb_df.iterrows():
        ds = row.get('dataset') or row.get('dataset_tag') or row.get('dataset_name')
        f1 = row.get('test_macro_f1') or row.get('macro_f1') or row.get('eval_macro_f1')
        if ds and f1:
            bb_row[ds] = f1
    if bb_row:
        pivot.loc['banglabert (nb05)'] = bb_row

fgm_path = TABLE_DIR / '06_fgm_banglabert_binary_summary_all_datasets.csv'
if fgm_path.exists():
    fgm_df = pd.read_csv(fgm_path)
    fgm_row = {}
    for _, row in fgm_df.iterrows():
        ds = row.get('dataset') or row.get('dataset_tag') or row.get('dataset_name')
        f1 = row.get('test_macro_f1') or row.get('macro_f1') or row.get('eval_macro_f1')
        if ds and f1:
            fgm_row[ds] = f1
    if fgm_row:
        pivot.loc['banglabert+fgm (nb06)'] = fgm_row

pivot['mean'] = pivot.mean(axis=1)
pivot = pivot.sort_values('mean', ascending=False)

print("="*90)
print("  MACRO-F1 COMPARISON — All Backbones x All Datasets (Test)")
print("="*90)
print(pivot.to_string(float_format='%.4f'))
print(f"\nBest backbone: {pivot['mean'].idxmax()} (mean={pivot['mean'].max():.4f})")

  MACRO-F1 COMPARISON — All Backbones x All Datasets (Test)
dataset                banglasarc3_binary  banglasarc_binary  ben_sarc_binary   mean
model_tag                                                                           
banglabert+fgm (nb06)              0.7880             0.9896           0.8022 0.8599
banglabert (nb05)                  0.7666             0.9834           0.8018 0.8506
muril                              0.7356             0.9771           0.7762 0.8296
xlm-roberta                        0.7306             0.9446           0.7613 0.8122
sagorsarker-bb                     0.7282             0.9487           0.7512 0.8094
mbert                              0.7280             0.9448           0.7496 0.8075

Best backbone: banglabert+fgm (nb06) (mean=0.8599)


In [10]:
# Save confusion matrices
cm_dict = {}
for _, row in results_df.iterrows():
    cm_dict[f"{row['model_tag']}_{row['dataset']}"] = json.loads(row['confusion_matrix'])
with open(TABLE_DIR / '07_multi_backbone_binary_confusion_matrices.json', 'w') as f:
    json.dump(cm_dict, f, indent=2)

# Save metadata
meta_cols = ['model_tag', 'model_name', 'dataset', 'train_samples', 'epochs', 'batch_size', 'lr', 'max_length', 'time_seconds']
results_df[meta_cols].to_csv(LOG_DIR / '07_multi_backbone_run_metadata.csv', index=False)

# Classification reports
all_reports = []
for _, row in results_df.iterrows():
    pp = PRED_DIR / f"07_{row['model_tag']}_{row['dataset']}_test_predictions.csv"
    if pp.exists():
        pdf = pd.read_csv(pp)
        rpt = classification_report(pdf['true_label'], pdf['pred_label'], output_dict=True, target_names=['Not Sarcastic', 'Sarcastic'])
        for cls_name, metrics in rpt.items():
            if isinstance(metrics, dict):
                all_reports.append({'model_tag': row['model_tag'], 'dataset': row['dataset'], 'class': cls_name, **metrics})
if all_reports:
    pd.DataFrame(all_reports).to_csv(REPORT_DIR / '07_multi_backbone_binary_classification_reports.csv', index=False)

print("All artifacts saved.")

All artifacts saved.


In [11]:
# Display confusion matrices
for _, row in results_df.iterrows():
    cm = np.array(json.loads(row['confusion_matrix']))
    print(f"\n{row['model_tag']} x {row['dataset']}")
    print(f"  Macro-F1: {row['test_macro_f1']:.4f} | Acc: {row['test_accuracy']:.4f}")
    print(f"                  Pred=0   Pred=1")
    print(f"    True=0     {cm[0,0]:>6}   {cm[0,1]:>6}")
    print(f"    True=1     {cm[1,0]:>6}   {cm[1,1]:>6}")


xlm-roberta x ben_sarc_binary
  Macro-F1: 0.7613 | Acc: 0.7613
                  Pred=0   Pred=1
    True=0        976      306
    True=1        306      976

xlm-roberta x banglasarc_binary
  Macro-F1: 0.9446 | Acc: 0.9473
                  Pred=0   Pred=1
    True=0        299       17
    True=1         10      186

xlm-roberta x banglasarc3_binary
  Macro-F1: 0.7306 | Acc: 0.7307
                  Pred=0   Pred=1
    True=0        286      115
    True=1        101      300

mbert x ben_sarc_binary
  Macro-F1: 0.7496 | Acc: 0.7500
                  Pred=0   Pred=1
    True=0        911      371
    True=1        270     1012

mbert x banglasarc_binary
  Macro-F1: 0.9448 | Acc: 0.9473
                  Pred=0   Pred=1
    True=0        297       19
    True=1          8      188

mbert x banglasarc3_binary
  Macro-F1: 0.7280 | Acc: 0.7282
                  Pred=0   Pred=1
    True=0        302       99
    True=1        119      282

sagorsarker-bb x ben_sarc_binary
  Macro-F1: 0.

In [12]:
print("Files produced in PROJECT/04_outputs:")
for p in sorted(PROJECT.rglob('07_*')):
    if p.is_file() and 'predictions' in str(p):
        print(f"  {p.relative_to(PROJECT)}  ({p.stat().st_size / 1e6:.1f} MB)")
print(f"\nDisk free (working): {disk_free_gb('/kaggle/working'):.1f} GB")
print(f"Disk free (tmp):     {disk_free_gb(BIG_TMP):.1f} GB")
print("\n✅ Notebook finished without exceeding disk limits.")

Files produced in PROJECT/04_outputs:
  04_outputs/predictions/07_mbert_banglasarc3_binary_test_predictions.csv  (0.2 MB)
  04_outputs/predictions/07_mbert_banglasarc3_binary_val_predictions.csv  (0.2 MB)
  04_outputs/predictions/07_mbert_banglasarc_binary_test_predictions.csv  (0.1 MB)
  04_outputs/predictions/07_mbert_banglasarc_binary_val_predictions.csv  (0.1 MB)
  04_outputs/predictions/07_mbert_ben_sarc_binary_test_predictions.csv  (0.6 MB)
  04_outputs/predictions/07_mbert_ben_sarc_binary_val_predictions.csv  (0.6 MB)
  04_outputs/predictions/07_muril_banglasarc3_binary_test_predictions.csv  (0.2 MB)
  04_outputs/predictions/07_muril_banglasarc3_binary_val_predictions.csv  (0.2 MB)
  04_outputs/predictions/07_muril_banglasarc_binary_test_predictions.csv  (0.1 MB)
  04_outputs/predictions/07_muril_banglasarc_binary_val_predictions.csv  (0.1 MB)
  04_outputs/predictions/07_muril_ben_sarc_binary_test_predictions.csv  (0.6 MB)
  04_outputs/predictions/07_muril_ben_sarc_binary_val_pr